# 00 数据准备
把 Excel 中的患者信息与配准影像按 MR 编号对应起来，再生成后面分类实验共用的四层缓存。

有完整数据的电脑运行这本即可。算力机已经有 `cohort.csv` 和 `resnet50_manual_cache.npz` 时，直接在 `config.json` 填好这两个路径，从 01 或 02 开始。原先生成的这两个文件可以继续使用。


## 路径与临床字段
`config.json` 只记录数据位置、Excel 列名、结果目录和时间分界日期。MoCA 用于生成结局标签：≤22 记为 1；它不作为模型输入。年龄、受教育年限、NIHSS、同型半胱氨酸和尿酸用于临床对照及融合模型。


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "config.json").exists() and (ROOT / "1117" / "config.json").exists():
    ROOT = ROOT / "1117"
cfg = json.loads((ROOT / "config.json").read_text(encoding="utf-8"))

def path_from_config(key):
    value = Path(cfg[key]).expanduser()
    return value if value.is_absolute() else ROOT / value

CACHE = path_from_config("cache_path")
COHORT = path_from_config("cohort_path")
RESULTS = path_from_config("results_dir")
SEED = 20260922
CLINICAL = ["age", "education", "nihss", "hcy", "ua"]
SEQUENCES = ["t1", "t2", "dwi"]
IMAGE_FILES = ["t1_flair.nii.gz", "t2.nii.gz", "dwi_b1000.nii.gz"]
print("患者清单：", COHORT)
print("结果目录：", RESULTS)


患者清单： e:\git\PSCI\outputs\resnet_baseline_001\data\cohort.csv
结果目录： e:\git\PSCI\results


## 读取影像与人工标注
下面按 MR 编号寻找包含配准三序列的目录，读取 T1-FLAIR、T2、DWI 和 `Untitled.nii.gz`。

三序列和标注需要相同尺寸、相近空间矩阵。矩阵方向元素允许 0.0001、平移元素允许 0.001 的数值误差；超出时提示核对，不自动移动图像。这只是检查文件是否共用网格，实际配准效果还要看叠图。

`max_layer` 沿用原代码：Excel 中从 1 开始的起始层，连取四层。标签 1、2、3 分别保存为三个通道；其临床含义以人工标注规则为准。其余层没有完整标注，不用于病灶训练。

`patients_root` 填直接包含各患者文件夹的目录；每例读取 `患者文件夹/配准图像`，患者文件夹名以 MR 编号结尾，例如 `姓名 MR250418095`。


In [2]:
import re
import nibabel as nib


def image_directories():
    root = path_from_config("patients_root")
    if not root.is_dir():
        raise FileNotFoundError(f"患者总目录不存在：{root}")

    folders = {}
    for patient_dir in sorted(root.iterdir()):
        if not patient_dir.is_dir():
            continue
        match = re.search(r"MR\s*(\d+)$", patient_dir.name.strip(), re.IGNORECASE)
        if not match:
            continue
        directory = patient_dir / "配准图像"
        if not (directory / "t1_flair.nii.gz").is_file():
            continue

        mr = "MR" + match.group(1)
        directory = directory.resolve()
        if mr in folders and folders[mr] != directory:
            raise ValueError(
                f"两个患者文件夹使用了同一个影像号 {mr}：\n"
                f"{folders[mr]}\n{directory}"
            )
        folders[mr] = directory

    if not folders:
        raise FileNotFoundError(
            f"未在 {root} 找到 患者文件夹/配准图像/t1_flair.nii.gz。"
            "请把 patients_root 设为直接包含各患者文件夹的目录。"
        )
    return folders


def read_four_slices(patient, folders=None):
    recorded = [Path(str(patient.get(key, ""))) for key in SEQUENCES + ["manual_mask"]]
    if not all(file.is_file() for file in recorded):
        folders = folders if folders is not None else image_directories()
        if patient.mr not in folders:
            raise FileNotFoundError(f"找不到{patient.mr}的三序列目录。")
        recorded = [folders[patient.mr] / name for name in IMAGE_FILES + ["Untitled.nii.gz"]]
    volumes = [nib.load(str(file)) for file in recorded]
    ref = volumes[0]
    tolerance = np.array([1e-4, 1e-4, 1e-4, 1e-3])
    if len(ref.shape) != 3 or any(v.shape != ref.shape or not np.allclose(v.affine, ref.affine, atol=tolerance, rtol=0) for v in volumes[1:]):
        raise ValueError(f"{patient.mr}三序列与标注的shape/affine不一致。")
    layer = float(patient.max_layer)
    if not np.isfinite(layer) or not layer.is_integer():
        raise ValueError(f"{patient.mr}起始层不是整数。")
    start = int(layer) - 1
    if start < 0 or start + 4 > ref.shape[2]:
        raise ValueError(f"{patient.mr}四层取层越界。")
    labels = np.asarray(volumes[-1].dataobj)
    if not np.isfinite(labels).all() or not set(np.unique(labels)) <= {0, 1, 2, 3}:
        raise ValueError(f"{patient.mr}标签必须为0/1/2/3。")
    if np.any(labels[:, :, :start]) or np.any(labels[:, :, start + 4:]):
        raise ValueError(f"{patient.mr}指定四层以外仍有标注，请核对起始层。")
    images = [np.asarray(v.dataobj, dtype=np.float32)[:, :, start:start + 4] for v in volumes[:3]]
    if any(not np.isfinite(image).all() for image in images):
        raise ValueError(f"{patient.mr}影像包含无效数值。")
    return images, labels[:, :, start:start + 4].astype(np.uint8)


def normalize_four(volume):
    lo, hi = np.percentile(volume, [1, 99.5])
    return np.clip((volume - lo) / max(float(hi - lo), 1e-6), 0, 1)


e:\anaconda\env\pytorch\Lib\site-packages\nibabel\optpkg.py:104: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 2.5.5)
  pkg = __import__(name, fromlist=fromlist)


## 匹配患者、划分队列、生成缓存
每个序列在选定四层内做 1%～99.5% 分位归一化，影像线性缩放到 224×224，标签用最近邻缩放。缓存格式与之前保持一致。

检查日期不晚于配置分界日的患者进入开发队列，之后进入时间测试队列。开发队列生成固定的患者级五折；后续所有任务都读取这份清单，不各自重新分组。临床缺失值暂时保留，到训练折内再填补。


In [3]:
from scipy.ndimage import zoom
from sklearn.model_selection import StratifiedKFold


def normalize_mr(value):
    text = str(value).strip().upper()
    match = re.fullmatch(r"(?:MR\s*)?(\d+)(?:\.0+)?", text)
    if not match:
        raise ValueError(f"无法识别影像号：{value}")
    return "MR" + match.group(1)


def make_cohort():
    source = pd.read_excel(path_from_config("excel_path"), sheet_name=cfg["sheet_name"])
    source.columns = source.columns.astype(str).str.strip()
    columns = cfg["columns"]
    required = ["mr", "moca", "max_layer", "check_time"] + CLINICAL
    missing = [columns[key] for key in required if columns[key] not in source.columns]
    if missing or not source.columns.is_unique:
        raise ValueError(f"请核对Excel列名；缺少：{missing}")
    cohort = pd.DataFrame({key: source[columns[key]] for key in required})
    cohort["moca"] = pd.to_numeric(cohort.moca, errors="coerce")
    usable = cohort.moca.between(0, 30)
    if not usable.all():
        print(f"排除{int((~usable).sum())}例缺失或无效MoCA。")
    cohort = cohort.loc[usable].copy()
    cohort["mr"] = cohort.mr.map(normalize_mr)
    if not cohort.mr.is_unique:
        raise ValueError("Excel中MR编号重复，请先确认每例患者使用哪次检查。")
    cohort["check_time"] = pd.to_datetime(cohort.check_time.astype(str).str.strip().str.replace(".", "-", regex=False),
                                           format="mixed", errors="coerce").dt.normalize()
    cohort["max_layer"] = pd.to_numeric(cohort.max_layer, errors="coerce")
    bad = cohort.check_time.isna() | ~np.isfinite(cohort.max_layer) | (cohort.max_layer % 1 != 0) | (cohort.max_layer < 1)
    if bad.any():
        raise ValueError(f"以下患者的检查日期或起始层有误：{cohort.loc[bad, 'mr'].tolist()}")
    cohort["max_layer"] = cohort.max_layer.astype(int)
    for key in CLINICAL:
        cohort[key] = pd.to_numeric(cohort[key], errors="coerce").replace([np.inf, -np.inf], np.nan)
    cohort["label"] = (cohort.moca <= 22).astype(int)
    cohort = cohort.sort_values(["check_time", "mr"], kind="mergesort").reset_index(drop=True)
    cohort["split"] = np.where(cohort.check_time <= pd.Timestamp(cfg["test_split_date"]), "development", "validation")
    cohort["development_fold"] = 0
    dev_index = np.flatnonzero(cohort.split.eq("development"))
    y = cohort.label.to_numpy()[dev_index]
    if np.bincount(y, minlength=2).min() < 5:
        raise ValueError("开发队列每类至少需要5例才能生成五折。")
    for fold, (_, valid) in enumerate(StratifiedKFold(5, shuffle=True, random_state=SEED).split(y, y), 1):
        cohort.loc[dev_index[valid], "development_fold"] = fold
    folders = image_directories()
    absent = [mr for mr in cohort.mr if mr not in folders]
    if absent:
        raise FileNotFoundError(f"以下患者未找到配准影像目录：{absent}")
    for key, filename in zip(SEQUENCES + ["manual_mask"], IMAGE_FILES + ["Untitled.nii.gz"]):
        cohort[key] = [str(folders[mr] / filename) for mr in cohort.mr]
    cohort["cache_index"] = np.arange(len(cohort))
    return cohort


def build_classification_cache(cohort):
    images = np.empty((len(cohort), 4, 3, 224, 224), dtype=np.float16)
    masks = np.zeros(images.shape, dtype=np.uint8)
    inspection = []
    for i, patient in cohort.iterrows():
        volumes, labels = read_four_slices(patient)
        normalized = [normalize_four(v) for v in volumes]
        h, w = labels.shape[:2]
        for s in range(4):
            for channel, volume in enumerate(normalized):
                images[i, s, channel] = zoom(volume[:, :, s], (224 / h, 224 / w), order=1, prefilter=False)
            for label in (1, 2, 3):
                masks[i, s, label - 1] = zoom((labels[:, :, s] == label).astype(np.uint8),
                                             (224 / h, 224 / w), order=0, prefilter=False)
        inspection.append({"mr": patient.mr, "first_slice": int(patient.max_layer), "last_slice": int(patient.max_layer) + 3,
                           **{f"label_{label}_pixels": int((labels == label).sum()) for label in (1, 2, 3)}})
        print(f"四层缓存 {i + 1}/{len(cohort)}：{patient.mr}", flush=True)
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    COHORT.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(CACHE, images=images, masks=masks, labels=cohort.label.to_numpy(dtype=np.int64),
                        mrs=cohort.mr.to_numpy(dtype=str), max_layers=cohort.max_layer.to_numpy(dtype=np.int16))
    cohort.to_csv(COHORT, index=False)
    pd.DataFrame(inspection).to_csv(COHORT.parent / "image_inspection.csv", index=False)
    return cohort


In [4]:
def read_cases():
    cases = pd.read_csv(COHORT, dtype={"mr": str})
    if not cases.mr.is_unique or not set(cases.split) <= {"development", "validation"}:
        raise ValueError("患者编号重复或分组有误，请检查cohort.csv。")
    if not set(cases.label) <= {0, 1} or cases.mr.isna().any():
        raise ValueError("患者编号缺失或结局不是0/1。")
    return cases


def read_classification_data():
    cases = read_cases()
    with np.load(CACHE, allow_pickle=False) as data:
        cache_mrs = data["mrs"].astype(str).tolist()
        if len(set(cache_mrs)) != len(cache_mrs) or set(cache_mrs) != set(cases.mr):
            raise ValueError("缓存与患者清单的MR编号不一致。")
        lookup = {mr: i for i, mr in enumerate(cache_mrs)}
        order = [lookup[mr] for mr in cases.mr]
        if not np.array_equal(data["labels"][order], cases.label) or not np.array_equal(data["max_layers"][order], cases.max_layer):
            raise ValueError("患者标签或取层与缓存不一致。")
        images = data["images"][order].astype(np.float32)
        masks = data["masks"][order].astype(np.uint8)
    if images.shape[1:] != (4, 3, 224, 224) or masks.shape != images.shape:
        raise ValueError("分类输入应为每例4层、3通道、224×224。")
    if not np.isfinite(images).all() or not np.isin(masks, [0, 1]).all():
        raise ValueError("影像存在无效值或mask不是0/1。")
    return cases, images, masks


## 生成或查看数据
第一次准备完整数据时将 `REBUILD_DATA` 改为 `True`。这会覆盖配置指定的缓存和清单；已有数据只想继续训练时保持 `False`。

输出的 `image_inspection.csv` 记录每位患者取了哪四层、各标签有多少像素。


In [ ]:
REBUILD_DATA = True  # 如果修改了Excel或影像数据，请将此改为True以重建缓存

if REBUILD_DATA:
    cases = make_cohort()
    display(cases.groupby("split").agg(n=("mr", "size"), positive=("label", "sum")))
    build_classification_cache(cases)
elif not COHORT.exists() or not CACHE.exists():
    print("尚未找到缓存。确认config.json中的数据路径后，将REBUILD_DATA改为True。")

if COHORT.exists() and CACHE.exists():
    cases, images, masks = read_classification_data()
    display(cases.groupby("split").agg(n=("mr", "size"), positive=("label", "sum")))
    display(cases[["mr", "moca", "label", "max_layer", "split", "development_fold"]].head())


## 看一例缓存
三行灰度图对应三种序列，最后一行把三个标签显示为红、绿、蓝。检查四层位置以及病灶与影像是否对得上。`EXAMPLE_MR` 留空时显示第一例。


In [ ]:
EXAMPLE_MR = ""

if COHORT.exists() and CACHE.exists():
    import matplotlib.pyplot as plt
    i = cases.index[cases.mr == EXAMPLE_MR][0] if EXAMPLE_MR else 0
    fig, axes = plt.subplots(4, 4, figsize=(10, 10))
    for s in range(4):
        for c, name in enumerate(SEQUENCES):
            axes[c, s].imshow(images[i, s, c], cmap="gray", vmin=0, vmax=1)
            axes[c, s].set_title(f"{name} | slice {int(cases.iloc[i].max_layer) + s}")
        axes[3, s].imshow(images[i, s, 0], cmap="gray", vmin=0, vmax=1)
        color = masks[i, s].transpose(1, 2, 0).astype(float)
        axes[3, s].imshow(np.dstack([color, color.max(axis=-1) * .6]))
    for ax in axes.flat:
        ax.axis("off")
    fig.suptitle(cases.iloc[i].mr)
    fig.tight_layout()
    plt.show()
